# MCP + LLM

```text
Usuario
  → LLM de OpenAI
  → Function calling
  → Router Python
  → FastMCP Client
  → MCP Server
  → Resultado
  → LLM redacta la respuesta final
```

Usaremos:

- `fastmcp` para construir y probar el servidor MCP.
- `openai` para llamar a un LLM usando la Responses API.
- Un cliente MCP in-memory para evitar levantar servidores externos durante la clase.


## 1. Instalación

Ejecutá esta celda si todavía no tenés las dependencias.


In [1]:
%pip install -U fastmcp openai python-dotenv



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Configurar API key y modelo

Este notebook usa la variable de entorno `OPENAI_API_KEY`.

Si no está cargada, te la va a pedir de forma oculta.


In [2]:
import os
import json
from getpass import getpass
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI
from fastmcp import FastMCP, Client

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

oai_client = OpenAI()

# Podés cambiarlo con:
# export OPENAI_MODEL="otro-modelo"
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano-2026-03-17")

MODEL



'gpt-5.4-nano-2026-03-17'

## 3. Datos simulados de una universidad

Para no depender de sistemas externos, usamos datos en memoria.


In [3]:
ALUMNOS: dict[str, dict[str, Any]] = {
    "A001": {
        "nombre": "Ana Torres",
        "carrera": "Ingeniería Informática",
        "email": "ana.torres@example.edu",
        "notas": [8, 9, 7],
        "entregas": {"TP1": True, "TP2": False, "TP3": True},
    },
    "A002": {
        "nombre": "Bruno López",
        "carrera": "Ingeniería Informática",
        "email": "bruno.lopez@example.edu",
        "notas": [6, 5, 7],
        "entregas": {"TP1": True, "TP2": True, "TP3": False},
    },
    "A003": {
        "nombre": "Carla Méndez",
        "carrera": "Ingeniería en Inteligencia Artificial",
        "email": "carla.mendez@example.edu",
        "notas": [10, 9, 10],
        "entregas": {"TP1": True, "TP2": True, "TP3": True},
    },
    "A004": {
        "nombre": "Diego Ruiz",
        "carrera": "Ingeniería Informática",
        "email": "diego.ruiz@example.edu",
        "notas": [3, 4, 5],
        "entregas": {"TP1": False, "TP2": False, "TP3": False},
    },
}

PROGRAMA_IA_GENERATIVA = {
    "materia": "IA Generativa",
    "codigo": "IA-402",
    "objetivos": [
        "Entender cómo se construyen aplicaciones con modelos de lenguaje.",
        "Diseñar herramientas, prompts y flujos de evaluación.",
        "Construir prototipos con APIs, RAG y MCP.",
    ],
    "unidades": [
        "Fundamentos de LLMs",
        "Prompting y evaluación",
        "RAG",
        "Agentes",
        "Model Context Protocol",
    ],
}

CALENDARIO_EXAMENES_2026 = {
    "parcial_1": "2026-05-12",
    "parcial_2": "2026-06-23",
    "recuperatorio": "2026-07-07",
    "entrega_final": "2026-07-14",
}


## 4. Construir el MCP Server

Creamos un servidor `universidad-mcp` y registramos tools, resources y prompts.


In [ ]:
from fastmcp import FastMCP, Client

mcp = FastMCP(
    name="universidad-mcp",
    instructions=(
        "Servidor MCP educativo para una clase introductoria. "
        "Usá tools solo cuando necesites datos reales de la universidad. "
        "Las acciones de alto impacto deben producir borradores y requerir aprobación humana."
    ),
)


## 5. Registrar Tools

Estas son las herramientas que el LLM podrá pedir ejecutar.


In [5]:
@mcp.tool
def buscar_alumno(legajo: str) -> dict[str, Any]:
    """
    Busca un alumno por legajo.

    Args:
        legajo: Identificador del alumno. Ejemplo: "A001".

    Returns:
        Datos públicos del alumno para la demo, incluyendo notas.
    """
    alumno = ALUMNOS.get(legajo)

    if alumno is None:
        return {
            "encontrado": False,
            "legajo": legajo,
            "mensaje": "No existe un alumno con ese legajo.",
        }

    return {
        "encontrado": True,
        "legajo": legajo,
        "nombre": alumno["nombre"],
        "carrera": alumno["carrera"],
        "email": alumno["email"],
        "notas": alumno["notas"],
    }


@mcp.tool
def calcular_promedio(notas: list[float]) -> dict[str, Any]:
    """
    Calcula el promedio de una lista de notas entre 0 y 10.

    Args:
        notas: Lista de notas numéricas. Ejemplo: [8, 7.5, 9].

    Returns:
        Promedio, cantidad de notas y estado de aprobación.
    """
    if not notas:
        raise ValueError("La lista de notas no puede estar vacía.")

    for nota in notas:
        if nota < 0 or nota > 10:
            raise ValueError("Todas las notas deben estar entre 0 y 10.")

    promedio = round(sum(notas) / len(notas), 2)

    return {
        "promedio": promedio,
        "cantidad_notas": len(notas),
        "estado": "aprobado" if promedio >= 4 else "desaprobado",
    }


@mcp.tool
def buscar_entregas_pendientes(materia: str, trabajo: str) -> dict[str, Any]:
    """
    Busca estudiantes que no entregaron un trabajo práctico.

    Args:
        materia: Nombre de la materia. En esta demo usar "IA Generativa".
        trabajo: Nombre del trabajo. Ejemplo: "TP2".

    Returns:
        Lista de estudiantes con entrega pendiente.
    """
    if materia.lower() != "ia generativa":
        raise ValueError("Esta demo solo tiene datos de la materia 'IA Generativa'.")

    pendientes = []

    for legajo, alumno in ALUMNOS.items():
        entrego = alumno["entregas"].get(trabajo)

        if entrego is False:
            pendientes.append(
                {
                    "legajo": legajo,
                    "nombre": alumno["nombre"],
                    "email": alumno["email"],
                }
            )

    return {
        "materia": materia,
        "trabajo": trabajo,
        "cantidad_pendientes": len(pendientes),
        "pendientes": pendientes,
    }


@mcp.tool
def crear_borrador_recordatorio_entrega(
    materia: str,
    trabajo: str,
    destinatarios: list[str],
    mensaje_base: str = "Les recordamos que tienen una entrega pendiente.",
) -> dict[str, Any]:
    """
    Crea un borrador de recordatorio de entrega. No envía emails.

    Esta tool está diseñada como una acción segura: prepara el texto,
    pero requiere revisión y aprobación humana antes de enviar.

    Args:
        materia: Nombre de la materia.
        trabajo: Nombre del trabajo pendiente.
        destinatarios: Lista de emails destinatarios.
        mensaje_base: Mensaje inicial para incluir en el borrador.

    Returns:
        Borrador listo para revisión humana.
    """
    if not destinatarios:
        raise ValueError("Debe haber al menos un destinatario.")

    asunto = f"Recordatorio de entrega pendiente: {trabajo} - {materia}"

    cuerpo = f"""Hola,

{mensaje_base}

Materia: {materia}
Trabajo: {trabajo}

Por favor, revisen la consigna y avísennos si tienen alguna dificultad.

Saludos,
Equipo docente
"""

    return {
        "tipo": "borrador",
        "requiere_aprobacion_humana": True,
        "asunto": asunto,
        "destinatarios": destinatarios,
        "cuerpo": cuerpo,
        "advertencia": "Este borrador NO fue enviado. Debe revisarlo una persona.",
    }


## 6. Registrar Resources y Prompts

Aunque el loop con OpenAI de este notebook se concentra en tools, dejamos resources y prompts para que el catálogo MCP sea completo.


In [6]:
@mcp.resource(
    "programa://materias/{materia_slug}",
    mime_type="application/json",
    annotations={"readOnlyHint": True, "idempotentHint": True},
)
def programa_materia(materia_slug: str) -> str:
    """Devuelve el programa de una materia."""
    if materia_slug != "ia-generativa":
        raise ValueError("Materia no disponible en esta demo.")

    return json.dumps(PROGRAMA_IA_GENERATIVA, ensure_ascii=False, indent=2)


@mcp.resource(
    "calendario://examenes/2026",
    mime_type="application/json",
    annotations={"readOnlyHint": True, "idempotentHint": True},
)
def calendario_examenes_2026() -> str:
    """Devuelve el calendario de exámenes 2026."""
    return json.dumps(CALENDARIO_EXAMENES_2026, ensure_ascii=False, indent=2)


@mcp.prompt
def feedback_entrega_tp(
    nombre_estudiante: str,
    trabajo: str,
    observaciones: str,
    tono: str = "cercano, claro y riguroso",
) -> str:
    """Genera una plantilla de feedback para una entrega práctica."""
    return f"""
Redactá feedback para {nombre_estudiante} sobre la entrega {trabajo}.

Tono: {tono}

Observaciones:
{observaciones}

Estructura obligatoria:
1. Empezá reconociendo algo positivo.
2. Señalá los problemas técnicos principales.
3. Explicá por qué esos problemas importan.
4. Proponé próximos pasos concretos.
5. Cerrá con una frase motivadora.
""".strip()


## 7. Probar el catálogo MCP

Antes de conectar el LLM, probamos que el servidor MCP exponga tools, resources y prompts.


In [7]:
async with Client(mcp) as mcp_client:
    await mcp_client.ping()
    tools = await mcp_client.list_tools()
    resources = await mcp_client.list_resources()
    prompts = await mcp_client.list_prompts()

print("TOOLS:")
for tool in tools:
    print("-", tool.name)

print("\nRESOURCES:")
for resource in resources:
    print("-", getattr(resource, "uri", resource))

print("\nPROMPTS:")
for prompt in prompts:
    print("-", prompt.name)


TOOLS:
- buscar_alumno
- calcular_promedio
- buscar_entregas_pendientes
- crear_borrador_recordatorio_entrega

RESOURCES:
- calendario://examenes/2026

PROMPTS:
- feedback_entrega_tp


## 8. Convertir Tools MCP a Tools de OpenAI

La Responses API recibe tools como JSON Schema.

Como FastMCP ya conoce el nombre, descripción y schema de entrada de cada tool, podemos convertir el catálogo MCP al formato de OpenAI.


In [8]:
def dump_model(obj: object) -> dict[str, Any]:
    """Convierte objetos Pydantic/modelos similares a dict de forma robusta."""
    if hasattr(obj, "model_dump"):
        return obj.model_dump(mode="json")
    if hasattr(obj, "dict"):
        return obj.dict()
    return obj.__dict__


def mcp_tool_to_openai_tool(tool: object) -> dict[str, Any]:
    """
    Convierte una tool de FastMCP/MCP al formato de function tool de OpenAI Responses API.
    """
    data = dump_model(tool)

    name = data.get("name") or getattr(tool, "name")
    description = (
        data.get("description")
        or data.get("annotations", {}).get("title")
        or f"Ejecuta la herramienta MCP {name}."
    )

    # Según versión, el schema puede venir como inputSchema, input_schema o parameters.
    parameters = (
        data.get("inputSchema")
        or data.get("input_schema")
        or data.get("parameters")
        or {
            "type": "object",
            "properties": {},
            "additionalProperties": False,
        }
    )

    # OpenAI espera una tool tipo function.
    return {
        "type": "function",
        "name": name,
        "description": description,
        "parameters": parameters,
    }


async with Client(mcp) as mcp_client:
    mcp_tools = await mcp_client.list_tools()

openai_tools = [mcp_tool_to_openai_tool(tool) for tool in mcp_tools]

print(json.dumps(openai_tools, ensure_ascii=False, indent=2))


[
  {
    "type": "function",
    "name": "buscar_alumno",
    "description": "Busca un alumno por legajo.",
    "parameters": {
      "additionalProperties": false,
      "properties": {
        "legajo": {
          "type": "string",
          "description": "Identificador del alumno. Ejemplo: \"A001\"."
        }
      },
      "required": [
        "legajo"
      ],
      "type": "object"
    }
  },
  {
    "type": "function",
    "name": "calcular_promedio",
    "description": "Calcula el promedio de una lista de notas entre 0 y 10.",
    "parameters": {
      "additionalProperties": false,
      "properties": {
        "notas": {
          "items": {
            "type": "number"
          },
          "type": "array",
          "description": "Lista de notas numéricas. Ejemplo: [8, 7.5, 9]."
        }
      },
      "required": [
        "notas"
      ],
      "type": "object"
    }
  },
  {
    "type": "function",
    "name": "buscar_entregas_pendientes",
    "description": "Bus

## 9. Router: ejecutar function calls usando FastMCP Client

Esta función es el puente:

```text
Function call del LLM
  → nombre + argumentos JSON
  → mcp_client.call_tool(...)
  → resultado JSON
  → function_call_output para el LLM
```


In [9]:
async def call_mcp_tool(tool_name: str, arguments: dict[str, Any]) -> str:
    """
    Ejecuta una tool MCP y devuelve un string JSON.
    La Responses API espera que el output de una function_call sea string.
    """
    try:
        async with Client(mcp) as mcp_client:
            result = await mcp_client.call_tool(tool_name, arguments)

        return json.dumps(
            {
                "ok": True,
                "tool": tool_name,
                "result": result.data,
            },
            ensure_ascii=False,
        )

    except Exception as exc:
        return json.dumps(
            {
                "ok": False,
                "tool": tool_name,
                "error": type(exc).__name__,
                "message": str(exc),
            },
            ensure_ascii=False,
        )


def has_function_calls(response) -> bool:
    return any(item.type == "function_call" for item in response.output)


## 10. Agent loop mínimo

Esta función hace varias vueltas:

1. Envía la pregunta del usuario al LLM con las tools disponibles.
2. Si el modelo pide ejecutar tools, las ejecutamos contra el MCP Server.
3. Devolvemos los resultados al modelo.
4. El modelo redacta una respuesta final.

Para clase, imprimimos cada tool call para que se vea la decisión del modelo.


In [10]:
SYSTEM_INSTRUCTIONS = """
Sos un asistente docente conectado a un MCP Server universitario.

Reglas:
- Usá tools cuando necesites datos reales de alumnos, notas o entregas.
- No inventes legajos, emails, notas ni entregas.
- Si una acción puede afectar a estudiantes, prepará borradores y pedí aprobación humana.
- Nunca afirmes que un email fue enviado; la tool disponible solo crea borradores.
- Respondé en español, de forma clara y didáctica.
""".strip()


async def run_llm_with_mcp(user_message: str, max_turns: int = 5) -> str:
    input_messages: list[Any] = [
        {
            "role": "user",
            "content": user_message,
        }
    ]

    for turn in range(max_turns):
        response = oai_client.responses.create(
            model=MODEL,
            instructions=SYSTEM_INSTRUCTIONS,
            input=input_messages,
            tools=openai_tools,
            tool_choice="auto",
        )

        # Guardamos la salida del modelo. Si hay function_call, esto es necesario
        # para que el siguiente request tenga el historial correcto.
        input_messages += response.output

        function_calls = [item for item in response.output if item.type == "function_call"]

        if not function_calls:
            return response.output_text

        print(f"\n--- Turno {turn + 1}: el modelo pidió {len(function_calls)} tool call(s) ---")

        for tool_call in function_calls:
            tool_name = tool_call.name
            args = json.loads(tool_call.arguments or "{}")

            print(f"🛠️ Tool call: {tool_name}")
            print(json.dumps(args, ensure_ascii=False, indent=2))

            tool_result = await call_mcp_tool(tool_name, args)

            print("📦 Resultado MCP:")
            print(json.dumps(json.loads(tool_result), ensure_ascii=False, indent=2))

            input_messages.append(
                {
                    "type": "function_call_output",
                    "call_id": tool_call.call_id,
                    "output": tool_result,
                }
            )

    raise RuntimeError("El loop alcanzó max_turns sin una respuesta final.")


## 11. Ejemplo 1: el LLM decide buscar un alumno y calcular su promedio

Acá el usuario pide algo de alto nivel. El modelo debería usar tools MCP para buscar datos reales y/o calcular.


In [11]:
respuesta = await run_llm_with_mcp(
    "Buscá al alumno A001, calculá su promedio y decime si está aprobado."
)

print("\nRESPUESTA FINAL:")
print(respuesta)



--- Turno 1: el modelo pidió 1 tool call(s) ---
🛠️ Tool call: buscar_alumno
{
  "legajo": "A001"
}
📦 Resultado MCP:
{
  "ok": true,
  "tool": "buscar_alumno",
  "result": {
    "encontrado": true,
    "legajo": "A001",
    "nombre": "Ana Torres",
    "carrera": "Ingeniería Informática",
    "email": "ana.torres@example.edu",
    "notas": [
      8,
      9,
      7
    ]
  }
}

--- Turno 2: el modelo pidió 1 tool call(s) ---
🛠️ Tool call: calcular_promedio
{
  "notas": [
    8,
    9,
    7
  ]
}
📦 Resultado MCP:
{
  "ok": true,
  "tool": "calcular_promedio",
  "result": {
    "promedio": 8.0,
    "cantidad_notas": 3,
    "estado": "aprobado"
  }
}

RESPUESTA FINAL:
Alumno **A001 (Ana Torres)** encontrado.

- **Notas:** 8, 9, 7  
- **Promedio:** **8.0**  
- **Estado:** **Aprobado** ✅


## 12. Ejemplo 2: el LLM busca entregas pendientes

El modelo debería llamar `buscar_entregas_pendientes`.


In [12]:
respuesta = await run_llm_with_mcp(
    "¿Quiénes no entregaron el TP2 de IA Generativa?"
)

print("\nRESPUESTA FINAL:")
print(respuesta)



--- Turno 1: el modelo pidió 1 tool call(s) ---
🛠️ Tool call: buscar_entregas_pendientes
{
  "materia": "IA Generativa",
  "trabajo": "TP2"
}
📦 Resultado MCP:
{
  "ok": true,
  "tool": "buscar_entregas_pendientes",
  "result": {
    "materia": "IA Generativa",
    "trabajo": "TP2",
    "cantidad_pendientes": 2,
    "pendientes": [
      {
        "legajo": "A001",
        "nombre": "Ana Torres",
        "email": "ana.torres@example.edu"
      },
      {
        "legajo": "A004",
        "nombre": "Diego Ruiz",
        "email": "diego.ruiz@example.edu"
      }
    ]
  }
}

RESPUESTA FINAL:
Estos son los estudiantes que **no entregaron el TP2 de IA Generativa**:

- **Ana Torres** — **Legajo A001** (ana.torres@example.edu)
- **Diego Ruiz** — **Legajo A004** (diego.ruiz@example.edu)


## 13. Ejemplo 3: capacidad vs control

En esta consulta, el usuario pide una acción sensible: enviar un recordatorio.

El diseño seguro es que la tool cree un **borrador**, no que envíe automáticamente.


In [13]:
respuesta = await run_llm_with_mcp(
    "Buscá quiénes no entregaron el TP2 de IA Generativa y prepará un recordatorio para enviarles. No lo envíes todavía."
)

print("\nRESPUESTA FINAL:")
print(respuesta)



--- Turno 1: el modelo pidió 1 tool call(s) ---
🛠️ Tool call: buscar_entregas_pendientes
{
  "materia": "IA Generativa",
  "trabajo": "TP2"
}
📦 Resultado MCP:
{
  "ok": true,
  "tool": "buscar_entregas_pendientes",
  "result": {
    "materia": "IA Generativa",
    "trabajo": "TP2",
    "cantidad_pendientes": 2,
    "pendientes": [
      {
        "legajo": "A001",
        "nombre": "Ana Torres",
        "email": "ana.torres@example.edu"
      },
      {
        "legajo": "A004",
        "nombre": "Diego Ruiz",
        "email": "diego.ruiz@example.edu"
      }
    ]
  }
}

--- Turno 2: el modelo pidió 1 tool call(s) ---
🛠️ Tool call: crear_borrador_recordatorio_entrega
{
  "materia": "IA Generativa",
  "trabajo": "TP2",
  "destinatarios": [
    "ana.torres@example.edu",
    "diego.ruiz@example.edu"
  ],
  "mensaje_base": "Les recordamos que tienen una entrega pendiente del TP2 de IA Generativa. Por favor, revisen la consigna y envíen el trabajo a la brevedad para poder regularizar su s

## 14. Ejemplo 4: cuando faltan datos

El modelo debería reconocer que necesita más información o usar la tool con los parámetros correctos.


In [14]:
respuesta = await run_llm_with_mcp(
    "Prepará un recordatorio para quienes no entregaron el trabajo, pero no te digo cuál."
)

print("\nRESPUESTA FINAL:")
print(respuesta)



--- Turno 1: el modelo pidió 1 tool call(s) ---
🛠️ Tool call: buscar_entregas_pendientes
{
  "materia": "IA Generativa",
  "trabajo": "TP1"
}
📦 Resultado MCP:
{
  "ok": true,
  "tool": "buscar_entregas_pendientes",
  "result": {
    "materia": "IA Generativa",
    "trabajo": "TP1",
    "cantidad_pendientes": 1,
    "pendientes": [
      {
        "legajo": "A004",
        "nombre": "Diego Ruiz",
        "email": "diego.ruiz@example.edu"
      }
    ]
  }
}

--- Turno 2: el modelo pidió 1 tool call(s) ---
🛠️ Tool call: crear_borrador_recordatorio_entrega
{
  "materia": "IA Generativa",
  "trabajo": "TP1",
  "destinatarios": [
    "diego.ruiz@example.edu"
  ],
  "mensaje_base": "Les recordamos que tienen una entrega pendiente del Trabajo Práctico correspondiente a la materia IA Generativa (TP1)."
}
📦 Resultado MCP:
{
  "ok": true,
  "tool": "crear_borrador_recordatorio_entrega",
  "result": {
    "tipo": "borrador",
    "requiere_aprobacion_humana": true,
    "asunto": "Recordatorio de 

## 15. ¿Qué acabamos de construir?

Construimos un patrón completo:

```text
LLM
  decide usar una tool
    ↓
OpenAI function calling
  devuelve nombre + argumentos
    ↓
Router Python
  traduce esa llamada a MCP
    ↓
FastMCP Client
  ejecuta mcp_client.call_tool(...)
    ↓
FastMCP Server
  corre la función real
    ↓
Resultado vuelve al LLM
  redacta la respuesta final
```

Este patrón es muy bueno para enseñar porque muestra que:

- MCP define capacidades.
- OpenAI function calling permite que el LLM pida usar capacidades.
- Nuestro código mantiene el control de ejecución.
- Las acciones sensibles pueden diseñarse como borradores o con aprobación humana.


## 16. Opcional: correr el MCP Server fuera del notebook

Desde una terminal:

```bash
fastmcp run universidad_mcp_server_openai_demo.py:mcp
```

Con HTTP:

```bash
fastmcp run universidad_mcp_server_openai_demo.py:mcp --transport http --port 8000
```

Y desde Python:

```python
from fastmcp import Client

async with Client("http://localhost:8000/mcp") as client:
    tools = await client.list_tools()
```


In [15]:
from fastmcp import Client

async with Client("http://localhost:8000/mcp") as client:
    tools = await client.list_tools()
    print(tools)

RuntimeError: Client failed to connect: All connection attempts failed

## 18. Ejercicios

1. Agregar una tool `estado_cursada(legajo: str)`.
2. Agregar una tool `crear_borrador_feedback(legajo: str, trabajo: str, observaciones: str)`.
3. Modificar el system prompt para que el modelo nunca muestre emails completos.
4. Agregar un parámetro `requiere_aprobacion=True` a las tools de alto impacto.
5. Hacer que el router bloquee tools de alto riesgo salvo que el usuario diga explícitamente “confirmo”.
